# Calculadora de Testes de Hipótese (Paramétricos)

Este notebook permite realizar testes de hipótese para médias, variâncias e proporções, tanto para uma quanto para duas populações.

### Como usar:
1. Execute a célula de **Setup**.
2. Execute a célula de **Funções de Teste** (mecanismo central).
3. Escolha o tipo de teste desejado nas seções abaixo, altere os parâmetros e execute (Shift+Enter).

## ⚙️ Setup

In [ ]:
import numpy as np
import scipy.stats as st

print("✅ Bibliotecas carregadas com sucesso!")

## 🛠️ Funções de Teste
Execute esta célula uma única vez para carregar todos os mecanismos de cálculo.

In [ ]:
def exibir_resultados(teste_nome, res):
    """Função auxiliar para imprimir os resultados de forma padronizada."""
    print(f"\n--- Resultados do {teste_nome} ---")
    print(f"Estatística de Teste: {res['estatistica_teste']:.4f}")
    print(f"P-valor (Valor de Prova): {res['p_valor']:.4f}")

    val_crit = res['valor_critico']
    if isinstance(val_crit, tuple):
        print(f"Valor Crítico (α={res['alfa']}, {res['tipo_teste']}): Inferior = {val_crit[0]:.4f}, Superior = {val_crit[1]:.4f}")
    else:
        print(f"Valor Crítico (α={res['alfa']}, {res['tipo_teste']}): {val_crit:.4f}")

    if 'graus_de_liberdade' in res: print(f"Graus de Liberdade: {res['graus_de_liberdade']:.2f}")
    if 'graus_de_liberdade_numerador' in res:
        print(f"GL (Numerador): {res['graus_de_liberdade_numerador']} | GL (Denominador): {res['graus_de_liberdade_denominador']}")
    
    status = "REJEITADA" if res['hipotese_rejeitada'] else "NÃO REJEITADA"
    print(f"\n👉 Conclusão: A hipótese nula (H₀) é {status}.")
    
    if res.get('power') is not None:
        print(f"\n--- Análise de Potência ---")
        print(f"Potência (1 - β): {res['power']:.4f}")
        print(f"β (Prob. Erro Tipo II): {1 - res['power']:.4f}")

def teste_qui_quadrado_variancia_uma_amostra(variancia_amostral, tamanho_amostra, variancia_pop_hipotetica, tipo_teste='duas_caudas', alfa=0.05, variancia_pop_alternativa=None):
    n, s2, sigma02 = tamanho_amostra, variancia_amostral, variancia_pop_hipotetica
    estatistica = (n - 1) * s2 / sigma02; df = n - 1
    if tipo_teste == 'duas_caudas':
        p_val = 2 * min(st.chi2.sf(estatistica, df), st.chi2.cdf(estatistica, df))
        val_crit = (st.chi2.ppf(alfa/2, df), st.chi2.ppf(1 - alfa/2, df))
    elif tipo_teste == 'cauda_esquerda':
        p_val = st.chi2.cdf(estatistica, df); val_crit = st.chi2.ppf(alfa, df)
    else: # cauda_direita
        p_val = st.chi2.sf(estatistica, df); val_crit = st.chi2.ppf(1 - alfa, df)
    
    power = None
    if variancia_pop_alternativa:
        nc = (n - 1) * (variancia_pop_alternativa / sigma02)
        if tipo_teste == 'duas_caudas': power = st.ncx2.cdf(val_crit[0], df, nc) + st.ncx2.sf(val_crit[1], df, nc)
        elif tipo_teste == 'cauda_esquerda': power = st.ncx2.cdf(val_crit, df, nc)
        else: power = st.ncx2.sf(val_crit, df, nc)
        
    res = {'estatistica_teste': estatistica, 'p_valor': p_val, 'hipotese_rejeitada': p_val < alfa, 'alfa': alfa, 'tipo_teste': tipo_teste, 'graus_de_liberdade': df, 'valor_critico': val_crit, 'power': power}
    exibir_resultados("Teste Qui-Quadrado (Variância/1 amostra)", res)

def teste_f_variancia_duas_amostras(var1, n1, var2, n2, tipo_teste='duas_caudas', alfa=0.05, ratio_alternativo=None):
    estatistica = var1 / var2; dfn, dfd = n1 - 1, n2 - 1
    if tipo_teste == 'duas_caudas':
        p_val = 2 * min(st.f.sf(estatistica, dfn, dfd), st.f.cdf(estatistica, dfn, dfd))
        val_crit = (st.f.ppf(alfa/2, dfn, dfd), st.f.ppf(1 - alfa/2, dfn, dfd))
    elif tipo_teste == 'cauda_esquerda':
        p_val = st.f.cdf(estatistica, dfn, dfd); val_crit = st.f.ppf(alfa, dfn, dfd)
    else: # cauda_direita
        p_val = st.f.sf(estatistica, dfn, dfd); val_crit = st.f.ppf(1 - alfa, dfn, dfd)
    
    power = None
    if ratio_alternativo:
        if tipo_teste == 'duas_caudas': power = st.ncf.cdf(val_crit[0], dfn, dfd, ratio_alternativo) + st.ncf.sf(val_crit[1], dfn, dfd, ratio_alternativo)
        elif tipo_teste == 'cauda_esquerda': power = st.ncf.cdf(val_crit, dfn, dfd, ratio_alternativo)
        else: power = st.ncf.sf(val_crit, dfn, dfd, ratio_alternativo)

    res = {'estatistica_teste': estatistica, 'p_valor': p_val, 'hipotese_rejeitada': p_val < alfa, 'alfa': alfa, 'tipo_teste': tipo_teste, 'graus_de_liberdade_numerador': dfn, 'graus_de_liberdade_denominador': dfd, 'valor_critico': val_crit, 'power': power}
    exibir_resultados("Teste F (Diferença de Variâncias)", res)

def teste_z_media_uma_amostra(media_amostral, n, mu0, sigma_pop, tipo_teste='duas_caudas', alfa=0.05, mu_H1=None):
    estatistica = (media_amostral - mu0) / (sigma_pop / np.sqrt(n))
    if tipo_teste == 'duas_caudas':
        p_val = 2 * st.norm.sf(abs(estatistica))
        val_crit = (st.norm.ppf(alfa/2), st.norm.ppf(1 - alfa/2))
    elif tipo_teste == 'cauda_esquerda':
        p_val = st.norm.cdf(estatistica); val_crit = st.norm.ppf(alfa)
    else: # cauda_direita
        p_val = st.norm.sf(estatistica); val_crit = st.norm.ppf(1 - alfa)
    
    power = None
    if mu_H1 is not None:
        shift = (mu_H1 - mu0) / (sigma_pop / np.sqrt(n))
        if tipo_teste == 'duas_caudas': power = st.norm.cdf(val_crit[0], loc=shift) + st.norm.sf(val_crit[1], loc=shift)
        elif tipo_teste == 'cauda_esquerda': power = st.norm.cdf(val_crit, loc=shift)
        else: power = st.norm.sf(val_crit, loc=shift)

    res = {'estatistica_teste': estatistica, 'p_valor': p_val, 'hipotese_rejeitada': p_val < alfa, 'alfa': alfa, 'tipo_teste': tipo_teste, 'valor_critico': val_crit, 'power': power}
    exibir_resultados("Teste Z (Média/1 amostra - σ conhecido)", res)

def teste_t_media_uma_amostra(media_amostral, n, mu0, s_amostral, tipo_teste='duas_caudas', alfa=0.05, mu_H1=None):
    estatistica = (media_amostral - mu0) / (s_amostral / np.sqrt(n)); df = n - 1
    if tipo_teste == 'duas_caudas':
        p_val = 2 * st.t.sf(abs(estatistica), df)
        val_crit = (st.t.ppf(alfa/2, df), st.t.ppf(1 - alfa/2, df))
    elif tipo_teste == 'cauda_esquerda':
        p_val = st.t.cdf(estatistica, df); val_crit = st.t.ppf(alfa, df)
    else: # cauda_direita
        p_val = st.t.sf(estatistica, df); val_crit = st.t.ppf(1 - alfa, df)
    
    power = None
    if mu_H1 is not None:
        nc = (mu_H1 - mu0) / (s_amostral / np.sqrt(n))
        if tipo_teste == 'duas_caudas': power = st.nct.cdf(val_crit[0], df, nc) + st.nct.sf(val_crit[1], df, nc)
        elif tipo_teste == 'cauda_esquerda': power = st.nct.cdf(val_crit, df, nc)
        else: power = st.nct.sf(val_crit, df, nc)

    res = {'estatistica_teste': estatistica, 'p_valor': p_val, 'hipotese_rejeitada': p_val < alfa, 'alfa': alfa, 'tipo_teste': tipo_teste, 'graus_de_liberdade': df, 'valor_critico': val_crit, 'power': power}
    exibir_resultados("Teste t (Média/1 amostra - σ desconhecido)", res)

def teste_z_media_duas_amostras(media1, n1, sigma1, media2, n2, sigma2, dif0=0, tipo_teste='duas_caudas', alfa=0.05, dif_H1=None):
    erro_padrao = np.sqrt(sigma1**2/n1 + sigma2**2/n2)
    estatistica = (media1 - media2 - dif0) / erro_padrao
    if tipo_teste == 'duas_caudas':
        p_val = 2 * st.norm.sf(abs(estatistica))
        val_crit = (st.norm.ppf(alfa/2), st.norm.ppf(1 - alfa/2))
    elif tipo_teste == 'cauda_esquerda':
        p_val = st.norm.cdf(estatistica); val_crit = st.norm.ppf(alfa)
    else: # cauda_direita
        p_val = st.norm.sf(estatistica); val_crit = st.norm.ppf(1 - alfa)

    power = None
    if dif_H1 is not None:
        shift = (dif_H1 - dif0) / erro_padrao
        if tipo_teste == 'duas_caudas': power = st.norm.cdf(val_crit[0], loc=shift) + st.norm.sf(val_crit[1], loc=shift)
        elif tipo_teste == 'cauda_esquerda': power = st.norm.cdf(val_crit, loc=shift)
        else: power = st.norm.sf(val_crit, loc=shift)
        
    res = {'estatistica_teste': estatistica, 'p_valor': p_val, 'hipotese_rejeitada': p_val < alfa, 'alfa': alfa, 'tipo_teste': tipo_teste, 'valor_critico': val_crit, 'power': power}
    exibir_resultados("Teste Z (Comparação de Médias - σ conhecidos)", res)

def teste_t_media_duas_amostras(media1, n1, s1, media2, n2, s2, dif0=0, var_iguais=False, tipo_teste='duas_caudas', alfa=0.05, dif_H1=None):
    if var_iguais:
        sp2 = ((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2); sp = np.sqrt(sp2)
        erro_padrao = sp * np.sqrt(1/n1 + 1/n2); df = n1 + n2 - 2
    else:
        erro_padrao = np.sqrt(s1**2/n1 + s2**2/n2)
        gl_num = (s1**2/n1 + s2**2/n2)**2
        gl_den = (s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1)
        df = gl_num / gl_den
    
    estatistica = (media1 - media2 - dif0) / erro_padrao
    if tipo_teste == 'duas_caudas':
        p_val = 2 * st.t.sf(abs(estatistica), df)
        val_crit = (st.t.ppf(alfa/2, df), st.t.ppf(1 - alfa/2, df))
    elif tipo_teste == 'cauda_esquerda':
        p_val = st.t.cdf(estatistica, df); val_crit = st.t.ppf(alfa, df)
    else: # cauda_direita
        p_val = st.t.sf(estatistica, df); val_crit = st.t.ppf(1 - alfa, df)
    
    power = None
    if dif_H1 is not None:
        effective_s = sp if var_iguais else np.sqrt(((s1**2/n1 + s2**2/n2) * (n1*n2)/(n1+n2)))
        nc = (dif_H1 - dif0) / erro_padrao
        if tipo_teste == 'duas_caudas': power = st.nct.cdf(val_crit[0], df, nc) + st.nct.sf(val_crit[1], df, nc)
        elif tipo_teste == 'cauda_esquerda': power = st.nct.cdf(val_crit, df, nc)
        else: power = st.nct.sf(val_crit, df, nc)
        
    res = {'estatistica_teste': estatistica, 'p_valor': p_val, 'hipotese_rejeitada': p_val < alfa, 'alfa': alfa, 'tipo_teste': tipo_teste, 'graus_de_liberdade': df, 'valor_critico': val_crit, 'power': power}
    exibir_resultados("Teste t (Welch ou agrupado - Comparação de Médias)", res)

def teste_t_pareado(media_dif, n, s_dif, mu_d0=0, tipo_teste='duas_caudas', alfa=0.05, mu_d_H1=None):
    estatistica = (media_dif - mu_d0) / (s_dif / np.sqrt(n)); df = n - 1
    if tipo_teste == 'duas_caudas':
        p_val = 2 * st.t.sf(abs(estatistica), df)
        val_crit = (st.t.ppf(alfa/2, df), st.t.ppf(1 - alfa/2, df))
    elif tipo_teste == 'cauda_esquerda':
        p_val = st.t.cdf(estatistica, df); val_crit = st.t.ppf(alfa, df)
    else: # cauda_direita
        p_val = st.t.sf(estatistica, df); val_crit = st.t.ppf(1 - alfa, df)
    
    res = {'estatistica_teste': estatistica, 'p_valor': p_val, 'hipotese_rejeitada': p_val < alfa, 'alfa': alfa, 'tipo_teste': tipo_teste, 'graus_de_liberdade': df, 'valor_critico': val_crit}
    exibir_resultados("Teste t Pareado (Diferença de Médias)", res)

def teste_z_proporcao_uma_amostra(p_amostral, n, p0, tipo_teste='duas_caudas', alfa=0.05):
    estatistica = (p_amostral - p0) / np.sqrt(p0*(1-p0)/n)
    if tipo_teste == 'duas_caudas':
        p_val = 2 * st.norm.sf(abs(estatistica))
        val_crit = (st.norm.ppf(alfa/2), st.norm.ppf(1 - alfa/2))
    elif tipo_teste == 'cauda_esquerda':
        p_val = st.norm.cdf(estatistica); val_crit = st.norm.ppf(alfa)
    else: # cauda_direita
        p_val = st.norm.sf(estatistica); val_crit = st.norm.ppf(1 - alfa)
    
    res = {'estatistica_teste': estatistica, 'p_valor': p_val, 'hipotese_rejeitada': p_val < alfa, 'alfa': alfa, 'tipo_teste': tipo_teste, 'valor_critico': val_crit}
    exibir_resultados("Teste Z (Proporção/1 amostra)", res)

def teste_z_proporcao_duas_amostras(p1, n1, p2, n2, dif0=0, tipo_teste='duas_caudas', alfa=0.05):
    p_pool = (p1*n1 + p2*n2) / (n1+n2)
    erro_padrao = np.sqrt(p_pool*(1-p_pool)*(1/n1 + 1/n2))
    estatistica = (p1 - p2 - dif0) / erro_padrao
    if tipo_teste == 'duas_caudas':
        p_val = 2 * st.norm.sf(abs(estatistica))
        val_crit = (st.norm.ppf(alfa/2), st.norm.ppf(1 - alfa/2))
    elif tipo_teste == 'cauda_esquerda':
        p_val = st.norm.cdf(estatistica); val_crit = st.norm.ppf(alfa)
    else: # cauda_direita
        p_val = st.norm.sf(estatistica); val_crit = st.norm.ppf(1 - alfa)
    
    res = {'estatistica_teste': estatistica, 'p_valor': p_val, 'hipotese_rejeitada': p_val < alfa, 'alfa': alfa, 'tipo_teste': tipo_teste, 'valor_critico': val_crit}
    exibir_resultados("Teste Z (Comparação de Proporções)", res)

## 1️⃣ Testes para UM Parâmetro
Use estas células quando quiser testar se a Média, Variância ou Proporção de uma população é igual a um valor hipotético.

### 1.a) Média (Teste t ou Z)

In [ ]:
# Escolha Z (σ conhecido) ou t (σ desconhecido)
teste_t_media_uma_amostra(
    media_amostral = 10.5,      # x̄ (média observada na amostra)
    n = 25,                     # número de elementos na amostra
    mu0 = 10.0,                 # μ₀ (média hipotética sob H0)
    s_amostral = 1.2,           # s (desvio padrão da amostra)
    tipo_teste = 'duas_caudas', # 'duas_caudas', 'cauda_esquerda' (<), 'cauda_direita' (>)
    alfa = 0.05,                # α (nível de significância)
    mu_H1 = 11.0                # μ₁ (média sob H1 para cálculo de potência/beta)
)

In [ ]:
# Caso conheça o σ populacional
teste_z_media_uma_amostra(
    media_amostral = 10.5,      # x̄
    n = 25,                     # n
    mu0 = 10.0,                 # μ₀
    sigma_pop = 1.0,            # σ (desvio padrão POPULACIONAL conhecido)
    tipo_teste = 'duas_caudas',
    alfa = 0.05,
    mu_H1 = 11.0
)

### 1.b) Variância (Qui-Quadrado)

In [ ]:
teste_qui_quadrado_variancia_uma_amostra(
    variancia_amostral = 0.8,       # s² (variância observada na amostra)
    tamanho_amostra = 20,           # n
    variancia_pop_hipotetica = 1.0, # σ²₀ (variância hipotética sob H0)
    tipo_teste = 'duas_caudas',     # 'duas_caudas', 'cauda_esquerda', 'cauda_direita'
    alfa = 0.05,                    # α
    variancia_pop_alternativa = 0.5 # σ²₁ (variância sob H1 para potência/beta)
)

### 1.c) Proporção (Teste Z)

In [ ]:
teste_z_proporcao_uma_amostra(
    p_amostral = 0.6,    # p̂ (proporção de sucessos observada na amostra)
    n = 100,             # tamanho da amostra (número de tentativas)
    p0 = 0.5,            # p₀ (proporção hipotética sob H0)
    tipo_teste = 'duas_caudas',
    alfa = 0.05
)

## 2️⃣ Testes para DOIS Parâmetros (Comparação)
Use estas células para comparar duas populações.

### 2.a) Comparação de Médias (Amostras Independentes)

In [ ]:
teste_t_media_duas_amostras(
    media1 = 12.4, n1 = 30, s1 = 1.5,   # média, n e DP da Amostra 1
    media2 = 11.2, n2 = 35, s2 = 1.8,   # média, n e DP da Amostra 2
    dif0 = 0,                           # Δ₀ (diferença hipotética μ1 - μ2, geralm. 0)
    var_iguais = False,                 # True: Variâncias Iguais | False: Welch (Varia. Dif.)
    tipo_teste = 'duas_caudas',         # 'duas_caudas', 'cauda_esquerda', 'cauda_direita'
    alfa = 0.05,                        # α
    dif_H1 = 0.5                        # Δ₁ (diferença real sob H1 para potência)
)

### 2.b) Comparação de Variâncias (Teste F)

In [ ]:
teste_f_variancia_duas_amostras(
    var1 = 1.4, n1 = 25,      # s² e n da Amostra 1
    var2 = 0.9, n2 = 28,      # s² e n da Amostra 2
    tipo_teste = 'duas_caudas', # 'duas_caudas', 'cauda_esquerda', 'cauda_direita'
    alfa = 0.05,               # α
    ratio_alternativo = 1.5   # σ²1/σ²2 Real sob H1 para cálculo de potência/beta
)

### 2.c) Comparação de Proporções (Teste Z)

In [ ]:
teste_z_proporcao_duas_amostras(
    p1 = 0.45, n1 = 200,    # p̂ e n da Amostra 1
    p2 = 0.38, n2 = 250,    # p̂ e n da Amostra 2
    dif0 = 0,               # p1 - p2 hipotético (H0)
    tipo_teste = 'duas_caudas',
    alfa = 0.05
)

### 2.d) Médias Pareadas (Amostras Dependentes)

In [ ]:
teste_t_pareado(
    media_dif = 1.2,    # d̄ (média das diferenças individuais di = xi - yi)
    n = 15,             # número de PARES na amostra
    s_dif = 0.5,        # sd (desvio padrão das diferenças observadas)
    mu_d0 = 0,          # μd hipotético (geralmente zero)
    tipo_teste = 'duas_caudas',
    alfa = 0.05
)